# Day 4 — Text embeddings + multimodal fusion

**Goal:** embed impression text with CLIP text encoder, fuse with image embeddings, build FAISS index, and measure improvement.

In [ ]:
!pip -q install pandas tqdm transformers torch faiss-cpu

In [ ]:

# === Setup ===
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, math, random, time
import numpy as np
import pandas as pd
from tqdm import tqdm

BASE = "/content/drive/MyDrive/mimic_project"
RAW  = f"{BASE}/raw"
DATA = f"{BASE}/dataset"
IMAGES = f"{DATA}/images"
REPORTS_DIR = f"{DATA}/reports"

os.makedirs(RAW, exist_ok=True)
os.makedirs(IMAGES, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print("BASE:", BASE)
print("RAW:", RAW)
print("IMAGES:", IMAGES)
print("REPORTS_DIR:", REPORTS_DIR)


## 1) Load dataset + image embeddings

In [ ]:

import faiss

df = pd.read_csv(f"{BASE}/clean_dataset_2696.csv")
image_emb = np.load(f"{BASE}/image_embeddings_2696.npy").astype("float32")
print("df:", df.shape, "image_emb:", image_emb.shape)


## 2) Load CLIP text encoder and embed impressions

In [ ]:

import torch
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name).to(device)
processor = CLIPProcessor.from_pretrained(model_name)
model.eval()

def embed_texts(texts, batch_size=32):
    out = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        inputs = processor(text=batch, return_tensors="pt", padding=True, truncation=True)
        inputs = {k:v.to(device) for k,v in inputs.items()}
        with torch.no_grad():
            feats = model.get_text_features(**inputs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        out.append(feats.cpu().numpy().astype("float32"))
    return np.vstack(out)

texts = df["impression"].astype(str).tolist()
text_emb = embed_texts(texts, batch_size=32)
print("text_emb:", text_emb.shape)


## 3) Fuse (alpha-weighted) and build FAISS index

In [ ]:

alpha = 0.5
fused = alpha * image_emb + (1 - alpha) * text_emb
fused = fused / np.linalg.norm(fused, axis=1, keepdims=True)
fused = fused.astype("float32")
print("Fused shape:", fused.shape, "Example norm:", float(np.linalg.norm(fused[0])))

dim = fused.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(fused)
print("Fused index size:", index.ntotal)

# quick sanity query
q = fused[0:1]
scores, idx = index.search(q, 5)
print("Query study_id:", int(df.loc[0, "study_id"]))
print("Retrieved study_ids:", [int(df.loc[i,"study_id"]) for i in idx[0]])
print("Scores:", scores[0])


## 4) Save fused embeddings and index (optional)

In [ ]:

np.save(f"{BASE}/fused_embeddings_alpha_0.5.npy", fused)
faiss.write_index(index, f"{BASE}/faiss_fused_alpha_0.5.index")
print("Saved fused embeddings + FAISS index to Drive.")


✅ **Day 4 deliverables:** `fused_embeddings_alpha_0.5.npy`, `faiss_fused_alpha_0.5.index`